# 03. 이상 소비 탐지 (Anomaly Detection)
## 개인 소비 분석 + 무의식 지출 탐지 AI

**목표:**
- 평소 소비 패턴을 학습하고, 이상한 지출을 자동 감지
- Isolation Forest 모델로 충동/이상 소비 후보 추출
- '왜 이상한가' 설명까지 출력

**활용 알고리즘:**
- `Isolation Forest` : 비지도 학습 기반 이상 탐지
- `Z-Score` : 금액 기반 통계적 이상치

## 0. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료!')

## 1. 데이터 로드

In [ ]:
df = pd.read_csv('../data/processed/spending_processed.csv')
df['date'] = pd.to_datetime(df['date'])

# hour 결측 처리
df['hour'] = df['hour'].fillna(12).astype(int)

print(f'데이터 shape: {df.shape}')
df.head(5)

## 2. 피처 엔지니어링
> 모델이 학습할 숫자 변수 만들기

In [ ]:
# 카테고리 → 숫자 인코딩
le = LabelEncoder()
df['category_enc'] = le.fit_transform(df['category'].astype(str))

# 시간대 구분 (0: 오전, 1: 오후, 2: 저녁, 3: 심야)
def time_zone(h):
    if h < 12:   return 0  # 오전
    elif h < 18: return 1  # 오후
    elif h < 22: return 2  # 저녁
    else:        return 3  # 심야

df['time_zone'] = df['hour'].apply(time_zone)

# 카테고리별 개인 평균 대비 이번 지출 비율
cat_mean = df.groupby('category')['amount'].transform('mean')
df['amount_vs_cat_mean'] = df['amount'] / cat_mean

# 같은 날 누적 지출
df['daily_cumsum'] = df.groupby('date')['amount'].cumsum()

# 주말 여부
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

print('피처 생성 완료!')
df[['amount', 'category_enc', 'time_zone', 'amount_vs_cat_mean', 'is_weekend', 'hour']].describe()

## 3. Z-Score 기반 이상치 (1차 탐지)

In [ ]:
# 전체 금액 기준 Z-Score
df['z_score'] = np.abs(stats.zscore(df['amount']))
df['is_zscore_anomaly'] = df['z_score'] > 2.0  # 2 시그마 이상

zscore_anomalies = df[df['is_zscore_anomaly']]
print(f'Z-Score 이상치: {len(zscore_anomalies)}건')
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 금액 분포
axes[0].hist(df['amount'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
for v in zscore_anomalies['amount']:
    axes[0].axvline(v, color='red', alpha=0.5, linewidth=1)
axes[0].set_title('금액 분포 (빨간선: Z-Score 이상치)', fontsize=12)
axes[0].set_xlabel('금액 (원)')

# 시계열 위에 이상치 표시
normal = df[~df['is_zscore_anomaly']]
axes[1].scatter(normal['date'], normal['amount'], color='steelblue', s=20, alpha=0.6, label='정상')
axes[1].scatter(zscore_anomalies['date'], zscore_anomalies['amount'],
                color='red', s=80, zorder=5, label='이상치')
axes[1].set_title('일별 지출 + 이상치 표시', fontsize=12)
axes[1].set_ylabel('금액 (원)')
axes[1].legend()
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('../data/processed/07_zscore_anomaly.png', dpi=120, bbox_inches='tight')
plt.show()

zscore_anomalies[['date', 'category', 'amount', 'memo', 'z_score']].sort_values('amount', ascending=False)

## 4. Isolation Forest (메인 탐지 모델)

In [ ]:
# 모델 입력 피처
FEATURES = ['amount', 'category_enc', 'time_zone', 'amount_vs_cat_mean', 'is_weekend', 'hour']

X = df[FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Isolation Forest 학습
# contamination: 전체 중 이상치 비율 예상값 (10%로 설정)
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.10,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_scaled)

# 예측: -1 = 이상치, 1 = 정상
df['iso_label'] = iso_forest.predict(X_scaled)
df['iso_score'] = iso_forest.score_samples(X_scaled)  # 낮을수록 이상
df['is_anomaly'] = df['iso_label'] == -1

anomaly_df = df[df['is_anomaly']]
print(f'Isolation Forest 이상 탐지: {len(anomaly_df)}건 / 전체 {len(df)}건')
print(f'이상치 비율: {len(anomaly_df)/len(df)*100:.1f}%')

## 5. 이상치 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

normal_df = df[~df['is_anomaly']]

# ① 시계열: 정상 vs 이상
axes[0, 0].scatter(normal_df['date'], normal_df['amount'], 
                   c='steelblue', s=25, alpha=0.6, label='정상 소비')
axes[0, 0].scatter(anomaly_df['date'], anomaly_df['amount'],
                   c='red', s=80, zorder=5, label='이상 소비', marker='^')
axes[0, 0].set_title('① 시계열: 이상 소비 탐지', fontsize=12)
axes[0, 0].set_ylabel('금액 (원)')
axes[0, 0].legend()
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=30)

# ② 시간대별 이상치 분포
zone_labels = {0: '오전', 1: '오후', 2: '저녁', 3: '심야'}
anomaly_zone = anomaly_df['time_zone'].map(zone_labels).value_counts()
total_zone = df['time_zone'].map(zone_labels).value_counts()
anomaly_ratio = (anomaly_zone / total_zone * 100).fillna(0)
colors_zone = ['#FF4444' if v > 15 else '#5B8CFF' for v in anomaly_ratio.values]
axes[0, 1].bar(anomaly_ratio.index, anomaly_ratio.values, color=colors_zone)
axes[0, 1].set_title('② 시간대별 이상치 비율 (%)', fontsize=12)
axes[0, 1].set_ylabel('이상치 비율 (%)')
axes[0, 1].axhline(10, color='red', linestyle='--', alpha=0.5, label='기준선(10%)')
axes[0, 1].legend()

# ③ 카테고리별 이상치 건수
anomaly_cat = anomaly_df['category'].value_counts()
axes[1, 0].bar(anomaly_cat.index, anomaly_cat.values, color='#FF6B6B')
axes[1, 0].set_title('③ 카테고리별 이상 소비 건수', fontsize=12)
axes[1, 0].set_ylabel('건수')
axes[1, 0].tick_params(axis='x', rotation=30)

# ④ 이상 점수 분포
axes[1, 1].hist(normal_df['iso_score'], bins=20, alpha=0.6, color='steelblue', label='정상')
axes[1, 1].hist(anomaly_df['iso_score'], bins=10, alpha=0.8, color='red', label='이상')
axes[1, 1].set_title('④ Isolation Forest 이상 점수 분포', fontsize=12)
axes[1, 1].set_xlabel('이상 점수 (낮을수록 이상)')
axes[1, 1].legend()

plt.suptitle('Isolation Forest 이상 소비 탐지 결과', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/08_isolation_forest.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. 이상치 '이유' 설명 출력
> 왜 이상한지 사람이 이해할 수 있는 언어로 설명

In [ ]:
def explain_anomaly(row, df):
    reasons = []

    # 1) 금액이 해당 카테고리 평균의 2배 이상
    cat_avg = df[df['category'] == row['category']]['amount'].mean()
    if row['amount'] > cat_avg * 2:
        reasons.append(f"'{row['category']}' 평균({cat_avg:,.0f}원)의 {row['amount']/cat_avg:.1f}배 지출")

    # 2) 심야 결제
    if row['hour'] >= 22:
        reasons.append(f"심야({row['hour']}시) 결제")

    # 3) 전체 평균의 3배 이상
    overall_avg = df['amount'].mean()
    if row['amount'] > overall_avg * 3:
        reasons.append(f"전체 평균({overall_avg:,.0f}원)의 {row['amount']/overall_avg:.1f}배")

    # 4) 감정 키워드
    impulse_kw = ['스트레스', '충동', '그냥', '질러', '기분', '새벽', '위로', '폭식']
    memo = str(row.get('memo', ''))
    hit = [kw for kw in impulse_kw if kw in memo]
    if hit:
        reasons.append(f"메모 감정 키워드: {', '.join(hit)}")

    if not reasons:
        reasons.append('복합적 패턴 이상 (모델 감지)')

    return ' / '.join(reasons)

anomaly_df = anomaly_df.copy()
anomaly_df['이상_이유'] = anomaly_df.apply(lambda r: explain_anomaly(r, df), axis=1)

print(f'총 이상 소비 {len(anomaly_df)}건\n')
display_cols = ['date', 'category', 'amount', 'hour', 'memo', '이상_이유']
display_cols = [c for c in display_cols if c in anomaly_df.columns]
anomaly_df[display_cols].sort_values('amount', ascending=False).reset_index(drop=True)

## 7. 절약 가능 금액 추정

In [ ]:
total_spend      = df['amount'].sum()
anomaly_spend    = anomaly_df['amount'].sum()
cat_avg_spend    = df.groupby('category')['amount'].transform('mean')
# 이상치를 카테고리 평균으로 대체했을 때 절약액
potential_saving = (anomaly_df['amount'] - df.loc[anomaly_df.index, 'amount'] /
                    anomaly_df['amount_vs_cat_mean']).clip(lower=0).sum()

print('=' * 50)
print('절약 가능 분석')
print('=' * 50)
print(f'총 지출:           {total_spend:>12,.0f}원')
print(f'이상 소비 합계:    {anomaly_spend:>12,.0f}원  ({anomaly_spend/total_spend*100:.1f}%)')
print(f'추정 절약 가능:    {potential_saving:>12,.0f}원')
print(f'절약 후 예상 지출: {total_spend - potential_saving:>12,.0f}원')
print('=' * 50)

## 8. 결과 저장

In [ ]:
# 이상치 플래그 포함한 전체 데이터 저장
df.to_csv('../data/processed/spending_with_anomaly.csv', index=False, encoding='utf-8-sig')
anomaly_df[display_cols + ['iso_score']].to_csv(
    '../data/processed/anomaly_list.csv', index=False, encoding='utf-8-sig'
)

print('저장 완료!')
print('  → data/processed/spending_with_anomaly.csv')
print('  → data/processed/anomaly_list.csv')
print()
print('다음 단계: 04_forecasting.ipynb (시계열 예측)')